In [2]:
import pandas as pd
from datasets import load_dataset, Dataset
from collections import Counter

In [18]:
df_difficulty = pd.read_json("../results_tag/Magpie-Llama-3.1-Pro-MT-300K-Filtered/difficulty_start-0_offset-300000.json")
df_quality = pd.read_json("../results_tag/Magpie-Llama-3.1-Pro-MT-300K-Filtered/quality_start-0_offset-300000.json")

In [14]:
df_base = load_dataset("Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filtered", split="train").to_pandas()

In [28]:
new_df = pd.DataFrame(
    {
        "instruction": df_base["instruction"],
        "conversations": df_base["conversations"],
        "model": df_base["model"],
        "quality": df_quality["quality"],
        "difficulty": df_difficulty["difficulty"],
        "source": ["Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filtered"]*len(df_base),
        "difficulty_generator": ["google/gemma-2-27b-it"]*len(df_base),
        "quality_generator": ["google/gemma-2-27b-it"]*len(df_base),
    }
)

In [29]:
new_df = new_df[new_df["quality"].isin({"good", "excellent"})]
new_df = new_df[new_df["difficulty"].isin({"hard", "medium", "very hard"})]

new_df = new_df.reset_index(drop=True)

new_df.head()

,instruction,conversations,model,quality,difficulty,source,difficulty_generator,quality_generator
0,A bag contains 50 balls of different colors : ...,"[{'from': 'human', 'value': 'A bag contains 50...",meta-llama/Meta-Llama-3.1-70B-Instruct,good,medium,Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filt...,google/gemma-2-27b-it,google/gemma-2-27b-it
1,The controlling shareholder of the company (th...,"[{'from': 'human', 'value': 'The controlling s...",meta-llama/Meta-Llama-3.1-70B-Instruct,good,medium,Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filt...,google/gemma-2-27b-it,google/gemma-2-27b-it
2,We have a square metal sheet of side 8. How mu...,"[{'from': 'human', 'value': 'We have a square ...",meta-llama/Meta-Llama-3.1-70B-Instruct,good,medium,Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filt...,google/gemma-2-27b-it,google/gemma-2-27b-it
3,What is meant by eigenvectors and eigenvalues ...,"[{'from': 'human', 'value': 'What is meant by ...",meta-llama/Meta-Llama-3.1-70B-Instruct,good,hard,Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filt...,google/gemma-2-27b-it,google/gemma-2-27b-it
4,Here is the beginning of a tutorial on malware...,"[{'from': 'human', 'value': 'Here is the begin...",meta-llama/Meta-Llama-3.1-70B-Instruct,good,medium,Magpie-Align/Magpie-Llama-3.1-Pro-MT-300K-Filt...,google/gemma-2-27b-it,google/gemma-2-27b-it


In [31]:
def fmt_shareGPT_to_openAI(conversations):
    messages = []
    for msg in conversations:
        if msg["from"] == "human":
            role = "user"
        elif msg["from"] == "gpt":
            role = "assistant"
        elif msg["from"] == "system":
            role = "system"
        else:
            raise ValueError(f"Invalid message from: {msg['from']}")
        
        messages.append({"role": role, "content": msg["value"]})
    
    return messages

new_df["messages"] = new_df["conversations"].apply(lambda x: fmt_shareGPT_to_openAI(x))

In [33]:
ds = Dataset.from_pandas(new_df)

In [35]:
ds.push_to_hub("slm-research-vn/Magpie-Llama-3.1-Pro-MT", private=True)

Uploading the dataset shards: 100%|██████████| 4/4 [00:52<00:00, 13.09s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/slm-research-vn/Magpie-Llama-3.1-Pro-MT/commit/9326974873073ce2a8a90cadfadf87a7b8707de1', commit_message='Upload dataset', commit_description='', oid='9326974873073ce2a8a90cadfadf87a7b8707de1', pr_url=None, pr_revision=None, pr_num=None)